In [ ]:
import os
import subprocess

# ── 0.1  Mount Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── 0.2  Persistent model cache on Drive (prevents re-downloading) ──
#    This MUST be set BEFORE any import of transformers / pyannote / etc.
STORAGE_ROOT = "/content/drive/MyDrive/Dubly_ME_Storage"
os.environ["HF_HOME"]       = f"{STORAGE_ROOT}/models"
os.environ["HF_HUB_CACHE"]  = f"{STORAGE_ROOT}/models/hub"
os.environ["TORCH_HOME"]    = f"{STORAGE_ROOT}/models/torch"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["TORCH_HOME"], exist_ok=True)

print(f"✔ HF_HOME       = {os.environ['HF_HOME']}")
print(f"✔ HF_HUB_CACHE  = {os.environ['HF_HUB_CACHE']}")
print(f"✔ TORCH_HOME    = {os.environ['TORCH_HOME']}")

# ── 0.3  Set HF_TOKEN from Colab Secrets (required for pyannote) ────
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✔ HF_TOKEN loaded from Colab Secrets.")
except Exception:
    print("⚠ Could not load HF_TOKEN from Colab Secrets.")
    print("  Set it manually:  os.environ['HF_TOKEN'] = 'hf_your_token'")


Mounted at /content/drive
✔ HF_HOME       = /content/drive/MyDrive/Dubly_ME_Storage/models
✔ HF_HUB_CACHE  = /content/drive/MyDrive/Dubly_ME_Storage/models/hub
✔ TORCH_HOME    = /content/drive/MyDrive/Dubly_ME_Storage/models/torch
✔ HF_TOKEN loaded from Colab Secrets.


In [ ]:
!pip install retina-face deep-sort-realtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 96.7 MB/s eta 0:00:00


# Face *tracking*

In [ ]:
from retinaface import RetinaFace
import numpy as np


class FaceDetector:
    def __init__(self, threshold: float = 0.9):
        self.threshold = threshold

    def detect(self, frame: np.ndarray):
        detections = []

        results = RetinaFace.detect_faces(frame)

        if not isinstance(results, dict):
            return detections

        for _, face_data in results.items():
            score = face_data["score"]

            if score < self.threshold:
                continue

            x1, y1, x2, y2 = face_data["facial_area"]

            detections.append({
                "bbox": [x1, y1, x2, y2],
                "confidence": float(score)
            })

        return detections

In [ ]:
from deep_sort_realtime.deepsort_tracker import DeepSort


class FaceTracker:
    def __init__(self):
        self.tracker = DeepSort(
            max_age=30,
            n_init=2,
            max_cosine_distance=0.4
        )

    def update(self, detections, frame):

        ds_detections = []

        for det in detections:
            x1, y1, x2, y2 = det["bbox"]

            width = x2 - x1
            height = y2 - y1

            ds_detections.append(
                (
                    [x1, y1, width, height],
                    det["confidence"],
                    "face"
                )
            )

        tracks = self.tracker.update_tracks(
            ds_detections,
            frame=frame
        )

        results = []

        for track in tracks:
            if not track.is_confirmed():
                continue

            track_id = track.track_id

            ltrb = track.to_ltrb()

            results.append({
                "track_id": track_id,
                "bbox": ltrb
            })

        return results

In [ ]:
import cv2


def draw_tracks(frame, tracks):

    for track in tracks:
        x1, y1, x2, y2 = map(int, track["bbox"])

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"ID:{track['track_id']}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    return frame

In [ ]:
import cv2
import os
import threading
import queue


class VideoReader:
    """
    A threaded, iterable video reader that yields frames from a video file.
    A background thread continuously reads frames into a queue while the
    main thread runs detection/tracking, overlapping I/O with GPU work.
    Works seamlessly with mounted Google Drive paths in Colab.
    """
    def __init__(self, video_path: str, queue_size: int = 128):
        if not os.path.exists(video_path):
            raise FileNotFoundError(f"Video file not found: {video_path}")

        self.cap = cv2.VideoCapture(video_path)

        if not self.cap.isOpened():
            raise IOError(f"Failed to open video at: {video_path}. File might be corrupted or format unsupported.")

        # cache these BEFORE the reader thread starts consuming frames
        self._fps = self.cap.get(cv2.CAP_PROP_FPS)
        self._frame_count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

        self._frame_queue = queue.Queue(maxsize=queue_size)
        self._stopped = threading.Event()

        self._thread = threading.Thread(target=self._reader_loop, daemon=True)
        self._thread.start()

    def _reader_loop(self):
        while not self._stopped.is_set():
            ret, frame = self.cap.read()
            if not ret:
                self._frame_queue.put(None)  # sentinel = end of video
                break
            self._frame_queue.put(frame)
        self.cap.release()

    def __iter__(self):
        return self

    def __next__(self):
        frame = self._frame_queue.get()
        if frame is None:
            raise StopIteration
        return frame

    def fps(self):
        return self._fps

    def frame_count(self):
        return self._frame_count

    def stop(self):
        self._stopped.set()

In [ ]:
import json
from pathlib import Path


class TrackManager:
    def __init__(self):
        self.tracks = []

    def add_tracks(self, frame_idx, tracks):

        for track in tracks:
            x1, y1, x2, y2 = track["bbox"]

            self.tracks.append({
                "frame_idx": int(frame_idx),
                "track_id": int(track["track_id"]),
                "bbox": [
                    float(x1),
                    float(y1),
                    float(x2),
                    float(y2)
                ]
            })

    def save(self, output_path):

        Path(output_path).parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with open(output_path, "w") as f:
            json.dump(
                self.tracks,
                f,
                indent=4
            )

In [ ]:
import cv2
from IPython.display import clear_output


class TrackingPipeline:
    def __init__(self, video_path):
        self.video_path = video_path
        self.detector = FaceDetector()
        self.tracker = FaceTracker()
        self.track_manager = TrackManager()

    def run(self):
        reader = VideoReader(self.video_path)
        frame_idx = 0

        try:
            for frame in reader:
                if frame.max() == 0:
                    frame_idx += 1
                    continue

                detections = self.detector.detect(frame)
                tracks = self.tracker.update(detections, frame)

                self.track_manager.add_tracks(frame_idx, tracks)
                frame_idx += 1

                if frame_idx % 30 == 0:
                    clear_output(wait=True)
                    print(f"Processed {frame_idx}/{reader.frame_count()} frames")
        finally:
            reader.stop()
            self.track_manager.save("/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/tracks.json")

In [ ]:
INPUT_MEDIA = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4"

pipeline = TrackingPipeline(
    video_path=INPUT_MEDIA
)

pipeline.run()

Processed 1260/1270 frames


# Active speakers

In [ ]:
!git clone -q https://github.com/TaoRuijie/TalkNet-ASD.git /content/TalkNet-ASD
!pip install -q gdown python_speech_features opencv-python-headless
!gdown --id 1AbN9fCf9IexMxEKXLQY2KYBlb-IhSEea -O /content/TalkNet-ASD/pretrain_TalkSet.model


fatal: destination path '/content/TalkNet-ASD' already exists and is not an empty directory.
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1AbN9fCf9IexMxEKXLQY2KYBlb-IhSEea
From (redirected): https://drive.google.com/uc?id=1AbN9fCf9IexMxEKXLQY2KYBlb-IhSEea&confirm=t&uuid=5a3d9a1c-9cb5-4443-82e1-fbda99b3a95d
To: /content/TalkNet-ASD/pretrain_TalkSet.model
100% 63.2M/63.2M [00:00<00:00, 224MB/s]


In [ ]:
import sys, os, json, glob, subprocess, math
from collections import defaultdict
import numpy as np
import cv2
import torch
from scipy.io import wavfile
from scipy import signal
import python_speech_features

sys.path.append("/content/TalkNet-ASD")
from talkNet import talkNet


VIDEO_PATH   = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4"
AUDIO_PATH   = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/orginal.wav"
TRACKS_JSON  = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/tracks.json"
WORK_DIR = "/content/asd_work" # ---------------------------------------------

PRETRAIN_MODEL = "/content/TalkNet-ASD/pretrain_TalkSet.model"
MIN_TRACK_LEN  = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

FRAMES_DIR = os.path.join(WORK_DIR, "frames")
CROP_DIR   = os.path.join(WORK_DIR, "crops")
os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print("Video FPS:", VIDEO_FPS)


Using device: cuda
Video FPS: 25.011324024656833


In [ ]:
CLEAN_AUDIO = os.path.join(WORK_DIR, "audio_16k.wav")
cmd = f'ffmpeg -y -i "{AUDIO_PATH}" -ac 1 -ar 16000 -vn "{CLEAN_AUDIO}" -loglevel panic'
subprocess.call(cmd, shell=True)
print("Wrote", CLEAN_AUDIO)


Wrote /content/asd_work/audio_16k.wav


In [ ]:
with open(TRACKS_JSON) as f:
    raw = json.load(f)

tracks_by_id = defaultdict(list)
for det in raw:
    tracks_by_id[det["track_id"]].append(det)

tracks = []
for tid, dets in tracks_by_id.items():
    dets.sort(key=lambda d: d["frame_idx"])
    frames = np.array([d["frame_idx"] for d in dets], dtype=int)
    bboxes = np.array([d["bbox"] for d in dets], dtype=float)
    if len(frames) < MIN_TRACK_LEN:
        print(f"Skipping track {tid}: only {len(frames)} frames (< {MIN_TRACK_LEN})")
        continue
    tracks.append({"track_id": tid, "frame": frames, "bbox": bboxes})

print(f"Loaded {len(tracks)} usable tracks out of {len(tracks_by_id)} total")


Loaded 2 usable tracks out of 2 total


In [ ]:
global_min = min(int(t["frame"].min()) for t in tracks)
global_max = max(int(t["frame"].max()) for t in tracks)

cap = cv2.VideoCapture(VIDEO_PATH)
idx, saved = 0, 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if global_min <= idx <= global_max:
        cv2.imwrite(os.path.join(FRAMES_DIR, f"{idx:07d}.jpg"), frame)
        saved += 1
    if idx > global_max:
        break
    idx += 1
cap.release()
print(f"Saved {saved} frames covering range [{global_min}, {global_max}]")


Saved 1269 frames covering range [1, 1269]


In [ ]:
CROPS_DIR = "crops_output"
os.makedirs(CROPS_DIR, exist_ok=True)

crop_paths = {}
crop_times = {}


VIDEO_FPS = 25

print(f"Starting to crop {len(tracks)} tracks...")

for t in tracks:
    tid = t["track_id"]

    out_prefix = os.path.join(CROPS_DIR, f"track_{tid:04d}")

    video_out, audio_out, t_out = crop_track(
        track=t,
        out_prefix=out_prefix,
        src_fps=VIDEO_FPS,
        crop_fps=CROP_FPS
    )

    crop_paths[tid] = (video_out, audio_out)
    crop_times[tid] = t_out

print("Cropping finished! The 'crop_paths' and 'crop_times' are now ready.")

Starting to crop 2 tracks...
Cropping finished! The 'crop_paths' and 'crop_times' are now ready.


In [ ]:
import numpy as np
from scipy import signal
import cv2
import os
import subprocess

CROP_FPS = 25

def crop_track(track, out_prefix, src_fps, crop_fps=CROP_FPS, crop_scale=0.40):
    frames = track["frame"].astype(float)
    bboxes = np.array(track["bbox"])  # x1, y1, x2, y2

    cx_raw = (bboxes[:, 0] + bboxes[:, 2]) / 2
    cy_raw = (bboxes[:, 1] + bboxes[:, 3]) / 2
    size_raw = np.maximum(bboxes[:, 2] - bboxes[:, 0], bboxes[:, 3] - bboxes[:, 1]) / 2

    k = len(frames) if len(frames) % 2 == 1 else len(frames) - 1
    k = max(min(k, 7), 1)
    if k >= 3:
        size_raw = signal.medfilt(size_raw, kernel_size=k)
        cx_raw = signal.medfilt(cx_raw, kernel_size=k)
        cy_raw = signal.medfilt(cy_raw, kernel_size=k)

    t_orig = frames / src_fps
    t_start, t_end = t_orig[0], t_orig[-1]

    n_out = max(int(np.floor((t_end - t_start) * crop_fps)) + 1, 1)
    t_out = t_start + np.arange(n_out) / crop_fps
    t_out = t_out[t_out <= t_end + 1e-6]
    if len(t_out) == 0:
        t_out = np.array([t_start])

    cx = np.interp(t_out, t_orig, cx_raw)
    cy = np.interp(t_out, t_orig, cy_raw)
    size = np.interp(t_out, t_orig, size_raw)
    src_frame_idx = np.round(t_out * src_fps).astype(int)

    avi_tmp = out_prefix + "_t.avi"
    vw = cv2.VideoWriter(avi_tmp, cv2.VideoWriter_fourcc(*"XVID"), crop_fps, (224, 224))

    for i in range(len(t_out)):
        img_path = os.path.join(FRAMES_DIR, f"{src_frame_idx[i]:07d}.jpg")
        image = cv2.imread(img_path)

        if image is None:
            vw.write(np.full((224, 224, 3), 110, dtype=np.uint8))
            continue

        img_h, img_w = image.shape[:2]
        cs = crop_scale
        bs = max(size[i], 1.0)

        cy_shifted = cy[i] + (bs * 0.15)

        bsi = int(bs * (1 + 2 * cs))
        padded = np.pad(image, ((bsi, bsi), (bsi, bsi), (0, 0)), "constant", constant_values=110)

        my, mx = cy_shifted + bsi, cx[i] + bsi

        y1 = int(my - bs * (1 + cs))
        y2 = int(my + bs * (1 + cs))
        x1 = int(mx - bs * (1 + cs))
        x2 = int(mx + bs * (1 + cs))

        face = padded[y1:y2, x1:x2]

        if face.size == 0 or face.shape[0] == 0 or face.shape[1] == 0:
            vw.write(np.full((224, 224, 3), 110, dtype=np.uint8))
            continue

        vw.write(cv2.resize(face, (224, 224)))
    vw.release()

    audio_start = t_start
    audio_end = t_start + len(t_out) / crop_fps
    audio_out = out_prefix + ".wav"

    cmd = f'ffmpeg -y -ss {audio_start:.3f} -to {audio_end:.3f} -i "{CLEAN_AUDIO}" -ac 1 -ar 16000 "{audio_out}" -loglevel panic'
    subprocess.call(cmd, shell=True)

    video_out = out_prefix + ".avi"
    cmd = f'ffmpeg -y -i "{avi_tmp}" -i "{audio_out}" -c:v copy -c:a copy "{video_out}" -loglevel panic'
    subprocess.call(cmd, shell=True)

    if os.path.exists(avi_tmp):
        os.remove(avi_tmp)

    return video_out, audio_out, t_out

In [ ]:
model = talkNet()
model.loadParameters(PRETRAIN_MODEL)
model = model.to(DEVICE)
import numpy as np
import cv2
import torch
import math
from scipy.io import wavfile
import python_speech_features

model.eval()

def score_track(video_path, audio_path, fps):
    sample_rate, audio = wavfile.read(audio_path)
    audio_feature = python_speech_features.mfcc(audio, sample_rate, numcep=13, winlen=0.025, winstep=0.010)

    cap = cv2.VideoCapture(video_path)
    video_feature = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        face = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        face = cv2.resize(face, (112, 112))
        video_feature.append(face)
    cap.release()
    video_feature = np.array(video_feature)

    if len(video_feature) == 0 or audio_feature.shape[0] == 0:
        return np.array([])


    min_video_len = min(video_feature.shape[0], audio_feature.shape[0] // 4)
    if min_video_len <= 0:
        return np.array([])

    video_feature = video_feature[:min_video_len, :, :]
    audio_feature = audio_feature[:min_video_len * 4, :]


    duration_list = [1, 2, 3, 4, 5, 6]
    all_scores = []

    with torch.no_grad():
        for duration in duration_list:
            batch_size = int(math.ceil(min_video_len / (duration * fps)))
            scores = []
            for i in range(batch_size):
                a_start = i * duration * 100
                a_end = min((i + 1) * duration * 100, audio_feature.shape[0])
                v_start = i * int(duration * fps)
                v_end = min((i + 1) * int(duration * fps), video_feature.shape[0])

                a_chunk = audio_feature[a_start:a_end, :]
                v_chunk = video_feature[v_start:v_end, :, :]

                if a_chunk.shape[0] == 0 or v_chunk.shape[0] == 0:
                    continue

                inputA = torch.FloatTensor(a_chunk).unsqueeze(0).to(DEVICE)
                inputV = torch.FloatTensor(v_chunk).unsqueeze(0).to(DEVICE)

                embedA = model.model.forward_audio_frontend(inputA)
                embedV = model.model.forward_visual_frontend(inputV)
                embedA, embedV = model.model.forward_cross_attention(embedA, embedV)
                out = model.model.forward_audio_visual_backend(embedA, embedV)

                score = model.lossAV.forward(out, labels=None)
                scores.extend(score)

            if scores:
                all_scores.append(scores)

    if not all_scores:
        return np.array([])

    min_len = min(len(s) for s in all_scores)
    all_scores = [s[:min_len] for s in all_scores]

    final_scores = np.round(np.mean(np.array(all_scores), axis=0), 3).astype(float)
    return final_scores

track_scores = {}
for t in tracks:
    v, a = crop_paths[t["track_id"]]
    t_out = crop_times[t["track_id"]]
    sc = score_track(v, a, CROP_FPS)
    n = min(len(sc), len(t_out))
    sc, t_out_trim = sc[:n], t_out[:n]

    if n == 0:
        track_scores[t["track_id"]] = {}
        print(f"Track {t['track_id']}: no scores produced (too short)")
        continue

    orig_frames = t["frame"]
    orig_times = orig_frames / VIDEO_FPS
    mapped_scores = np.interp(orig_times, t_out_trim, sc, left=sc[0], right=sc[-1])

    track_scores[t["track_id"]] = {int(f): float(s) for f, s in zip(orig_frames, mapped_scores)}
    print(f"Track {t['track_id']}: {n} scored @25fps -> mapped to {len(orig_frames)} original frames")


07-09 19:00:07 Model para number = 15.01
Track 1: 43 scored @25fps -> mapped to 44 original frames
Track 2: 1268 scored @25fps -> mapped to 1269 original frames


In [ ]:
import json
import os
import numpy as np
from collections import defaultdict
from scipy.ndimage import median_filter


SPEAKER_THRESHOLD = 0.5

def smooth_scores(fs_dict, window=5):
    smoothed = {}
    for tid, fs in fs_dict.items():
        frames_sorted = sorted(fs.keys())
        if not frames_sorted:
            continue

        vals = np.array([fs[f] for f in frames_sorted])


        sm = median_filter(vals, size=window)

        smoothed[tid] = {f: float(v) for f, v in zip(frames_sorted, sm)}
    return smoothed

track_scores_smoothed = smooth_scores(track_scores, window=5)

frame_scores = defaultdict(dict)
for tid, fs in track_scores_smoothed.items():
    for fidx, sc in fs.items():
        frame_scores[fidx][tid] = sc

active_speaker_per_frame = {}
for fidx, scores_here in frame_scores.items():

    active_tids = [tid for tid, score in scores_here.items() if score > SPEAKER_THRESHOLD]

    best_tid = max(scores_here, key=scores_here.get) if scores_here else None
    best_score = scores_here[best_tid] if best_tid is not None else 0.0

    final_active_tid = int(best_tid) if best_score > SPEAKER_THRESHOLD else None

    active_speaker_per_frame[fidx] = {
        "active_track_id": final_active_tid,
        "all_active_tracks": [int(t) for t in active_tids], # قائمة بكل من يتحدث
        "scores": {int(k): float(v) for k, v in scores_here.items()},
    }

output_records = []
for det in raw:
    fidx, tid = det["frame_idx"], det["track_id"]
    score = frame_scores.get(fidx, {}).get(tid)

    is_active = False
    if fidx in active_speaker_per_frame:
        is_active = tid in active_speaker_per_frame[fidx]["all_active_tracks"]

    output_records.append({
        **det,
        "asd_score": score,
        "is_active_speaker": bool(is_active)
    })

OUTPUT_JSON = os.path.join(WORK_DIR, "active_speaker_tracks.json")
with open(OUTPUT_JSON, "w") as f:
    json.dump(output_records, f, indent=2)

ACTIVE_PER_FRAME_JSON = os.path.join(WORK_DIR, "active_speaker_per_frame.json")
with open(ACTIVE_PER_FRAME_JSON, "w") as f:
    json.dump({str(k): v for k, v in sorted(active_speaker_per_frame.items())}, f, indent=2)

print("Saved:", OUTPUT_JSON)
print("Saved:", ACTIVE_PER_FRAME_JSON)

Saved: /content/asd_work/active_speaker_tracks.json
Saved: /content/asd_work/active_speaker_per_frame.json


# Post Processing After Talknet

In [ ]:
import os, json, math
from collections import defaultdict
import numpy as np
import cv2
from scipy import signal

TRACKS_JSON = "/content/drive/MyDrive/Dubly_ME_Storage/data/asd_work/active_speaker_tracks.json"
VIDEO_PATH  = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4"
WORK_DIR    = "/content/drive/MyDrive/Dubly_ME_Storage/data/asd_work"

SMOOTH_WINDOW_SEC   = 0.20
MIN_SEGMENT_SEC     = 0.30
MERGE_GAP_SEC       = 0.15
MIN_ACTIVE_SCORE    = 0.0

os.makedirs(WORK_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print("Video FPS:", VIDEO_FPS)

def sec_to_frames(sec, min_val=1):
    n = int(round(sec * VIDEO_FPS))
    return max(n, min_val)

SMOOTH_WINDOW_FRAMES = sec_to_frames(SMOOTH_WINDOW_SEC)
if SMOOTH_WINDOW_FRAMES % 2 == 0:
    SMOOTH_WINDOW_FRAMES += 1
MIN_SEGMENT_FRAMES = sec_to_frames(MIN_SEGMENT_SEC)
MERGE_GAP_FRAMES = sec_to_frames(MERGE_GAP_SEC, min_val=0)

print(f"Smoothing window: {SMOOTH_WINDOW_FRAMES} frames")
print(f"Min segment length: {MIN_SEGMENT_FRAMES} frames")
print(f"Merge gap: {MERGE_GAP_FRAMES} frames")


Video FPS: 25.011324024656833
Smoothing window: 5 frames
Min segment length: 8 frames
Merge gap: 4 frames


In [ ]:
TRACKS_JSON = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/active_speaker_tracks.json"

with open(TRACKS_JSON) as f:
    raw = json.load(f)

by_track = defaultdict(list)
for det in raw:
    by_track[det["track_id"]].append(det)

tracks = {}
for tid, dets in by_track.items():
    dets.sort(key=lambda d: d["frame_idx"])
    frame = np.array([d["frame_idx"] for d in dets], dtype=int)
    score = np.array([d.get("asd_score") if d.get("asd_score") is not None else -999.0 for d in dets], dtype=float)
    active = (score > MIN_ACTIVE_SCORE).astype(float)
    bbox = np.array([d["bbox"] for d in dets], dtype=float)
    tracks[tid] = {"frame": frame, "score": score, "active": active, "bbox": bbox}

print(f"Loaded {len(tracks)} tracks: " + ", ".join(f"track {t} ({len(v['frame'])} frames)" for t, v in tracks.items()))


Loaded 2 tracks: track 1 (44 frames), track 2 (1269 frames)


In [ ]:
for tid, t in tracks.items():
    n = len(t["active"])
    k = min(SMOOTH_WINDOW_FRAMES, n if n % 2 == 1 else n - 1)
    k = max(k, 1)
    if k >= 3:
        t["active_smoothed"] = signal.medfilt(t["active"], kernel_size=k)
    else:
        t["active_smoothed"] = t["active"].copy()

frame_candidates = defaultdict(list)
for tid, t in tracks.items():
    for f, a, s in zip(t["frame"], t["active_smoothed"], t["score"]):
        if a > 0.5:
            frame_candidates[f].append((tid, s))

resolved_active_track = {}
for f, cands in frame_candidates.items():
    best_tid = max(cands, key=lambda x: x[1])[0]
    resolved_active_track[f] = best_tid

for tid, t in tracks.items():
    final = []
    for f, a in zip(t["frame"], t["active_smoothed"]):
        final.append(1.0 if (a > 0.5 and resolved_active_track.get(f) == tid) else 0.0)
    t["active_final"] = np.array(final)

print("Smoothing done.")


Smoothing done.


In [ ]:
def build_segments(frame, active_final, score):
    """Group consecutive active=1 runs into segments, then merge close segments and drop short ones."""
    segments = []
    i = 0
    n = len(frame)
    while i < n:
        if active_final[i] > 0.5:
            j = i
            while j + 1 < n and active_final[j + 1] > 0.5:
                j += 1
            segments.append({
                "start_frame": int(frame[i]),
                "end_frame": int(frame[j]),
                "avg_score": float(np.mean(score[i:j + 1])),
            })
            i = j + 1
        else:
            i += 1

    if not segments:
        return []

    merged = [segments[0]]
    for seg in segments[1:]:
        prev = merged[-1]
        if seg["start_frame"] - prev["end_frame"] - 1 <= MERGE_GAP_FRAMES:
            prev["end_frame"] = seg["end_frame"]
            prev["avg_score"] = (prev["avg_score"] + seg["avg_score"]) / 2
        else:
            merged.append(seg)

    final = [s for s in merged if (s["end_frame"] - s["start_frame"] + 1) >= MIN_SEGMENT_FRAMES]
    return final


all_segments = []
for tid, t in tracks.items():
    segs = build_segments(t["frame"], t["active_final"], t["score"])
    for s in segs:
        s["track_id"] = tid
        s["start_time"] = round(s["start_frame"] / VIDEO_FPS, 3)
        s["end_time"] = round((s["end_frame"] + 1) / VIDEO_FPS, 3)
        s["duration_frames"] = s["end_frame"] - s["start_frame"] + 1
        s["avg_score"] = round(s["avg_score"], 3)
    all_segments.extend(segs)

all_segments.sort(key=lambda s: s["start_frame"])
print(f"Generated {len(all_segments)} speaking segments across {len(tracks)} tracks")
for s in all_segments[:10]:
    print(s)


Generated 12 speaking segments across 2 tracks
{'start_frame': 26, 'end_frame': 53, 'avg_score': 0.594, 'track_id': 2, 'start_time': 1.04, 'end_time': 2.159, 'duration_frames': 28}
{'start_frame': 83, 'end_frame': 100, 'avg_score': 1.13, 'track_id': 2, 'start_time': 3.318, 'end_time': 4.038, 'duration_frames': 18}
{'start_frame': 125, 'end_frame': 150, 'avg_score': 1.504, 'track_id': 2, 'start_time': 4.998, 'end_time': 6.037, 'duration_frames': 26}
{'start_frame': 164, 'end_frame': 294, 'avg_score': 2.217, 'track_id': 2, 'start_time': 6.557, 'end_time': 11.795, 'duration_frames': 131}
{'start_frame': 301, 'end_frame': 330, 'avg_score': 1.755, 'track_id': 2, 'start_time': 12.035, 'end_time': 13.234, 'duration_frames': 30}
{'start_frame': 351, 'end_frame': 411, 'avg_score': 1.452, 'track_id': 2, 'start_time': 14.034, 'end_time': 16.473, 'duration_frames': 61}
{'start_frame': 426, 'end_frame': 526, 'avg_score': 1.351, 'track_id': 2, 'start_time': 17.032, 'end_time': 21.07, 'duration_frame

In [ ]:
ACTIVE_SPEAKERS_JSON = os.path.join(WORK_DIR, "active_speakers.json")
with open(ACTIVE_SPEAKERS_JSON, "w") as f:
    json.dump(all_segments, f, indent=2)
print("Saved:", ACTIVE_SPEAKERS_JSON)


Saved: /content/drive/MyDrive/Dubly_ME_Storage/data/asd_work/active_speakers.json


In [ ]:
ANNOTATED_VIDEO = os.path.join(WORK_DIR, "annotated_preview.mp4")

active_frame_ranges = defaultdict(list)  # track_id -> [(start,end), ...]
for s in all_segments:
    active_frame_ranges[s["track_id"]].append((s["start_frame"], s["end_frame"]))

def is_active_at(tid, fidx):
    for start, end in active_frame_ranges.get(tid, []):
        if start <= fidx <= end:
            return True
    return False

frame_lookup = defaultdict(list)
for det in raw:
    frame_lookup[det["frame_idx"]].append((det["track_id"], det["bbox"]))

global_min = min(int(t["frame"].min()) for t in tracks.values())
global_max = max(int(t["frame"].max()) for t in tracks.values())

cap = cv2.VideoCapture(VIDEO_PATH)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(ANNOTATED_VIDEO, fourcc, VIDEO_FPS, (w, h))

idx = 0
written = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if idx < global_min:
        idx += 1
        continue
    if idx > global_max:
        break

    for tid, bbox in frame_lookup.get(idx, []):
        x1, y1, x2, y2 = [int(round(v)) for v in bbox]
        active = is_active_at(tid, idx)
        color = (0, 255, 0) if active else (0, 0, 255)  # BGR: green if active, red otherwise
        thickness = 3 if active else 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
        label = f"track {tid}" + (" SPEAKING" if active else "")
        cv2.putText(frame, label, (x1, max(y1 - 8, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    writer.write(frame)
    written += 1
    idx += 1

cap.release()
writer.release()
print(f"Wrote {written} annotated frames -> {ANNOTATED_VIDEO}")


Wrote 1269 annotated frames -> /content/drive/MyDrive/Dubly_ME_Storage/data/asd_work/annotated_preview.mp4


# Preparing for lip sync

In [ ]:
def overlap_duration(start1, end1, start2, end2):
    """
    Returns overlap duration in seconds.
    """
    return max(0, min(end1, end2) - max(start1, start2))

In [ ]:
import json

with open("/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/active_speakers.json", "r") as f:
    active_segments = json.load(f)

with open("/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/translation.json", "r", encoding="utf-8") as f:
    translation = json.load(f)

segments = translation["segments"]

In [ ]:
mapping = []

for active in active_segments:

    best_overlap = 0
    best_segment = None

    for seg in segments:

        overlap = overlap_duration(
            active["start_time"],
            active["end_time"],
            seg["start_time"],
            seg["end_time"]
        )

        if overlap > best_overlap:
            best_overlap = overlap
            best_segment = seg

    if best_segment is None:
        continue

    mapping.append({
        "track_id": active["track_id"],
        "speaker_id": best_segment["speaker_id"],
        "segment_id": best_segment["segment_id"],

        "video_start": active["start_time"],
        "video_end": active["end_time"],

        "audio_start": best_segment["start_time"],
        "audio_end": best_segment["end_time"],

        "overlap": round(best_overlap,3)
    })

In [ ]:
with open("/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/track_speaker_mapping.json","w") as f:
    json.dump(mapping,f,indent=4)

NameError: name 'mapping' is not defined

In [ ]:
import os
import json
import subprocess

# Paths
MAPPING_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/track_speaker_mapping.json"
AUDIO_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/final_dubbed.wav"

OUTPUT_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# Load track-speaker mapping
with open(MAPPING_FILE, "r", encoding="utf-8") as f:
    track_mapping = json.load(f)


for item in track_mapping:

    track_id = item["track_id"]
    segment_id = item["segment_id"]

    # output filename
    out_file = os.path.join(
        OUTPUT_DIR,
        f"track_{track_id}_segment_{segment_id}.wav"
    )

    # Use the audio timeline from mapping
    start_time = item["audio_start"]
    end_time = item["audio_end"]

    cmd = [
        "ffmpeg",
        "-y",
        "-i",
        AUDIO_FILE,
        "-ss",
        str(start_time),
        "-to",
        str(end_time),
        "-ac",
        "1",
        "-ar",
        "16000",
        out_file,
        "-loglevel",
        "error"
    ]

    subprocess.run(cmd, check=True)

    print(
        f"Created: {out_file} "
        f"({start_time:.3f}s -> {end_time:.3f}s)"
    )


print("All audio clips extracted successfully.")

Created: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_1.wav (0.900s -> 3.200s)
Created: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_2.wav (3.200s -> 4.300s)
Created: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_3.wav (4.800s -> 6.300s)
Created: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_4.wav (6.600s -> 9.900s)
Created: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_5.wav (9.900s -> 13.600s)
Created: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_6.wav (13.900s -> 16.800s)
Created: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_7.wav (17.000s -> 25.400s)
Created: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_7.wav (17.000s -> 25.400s)
Created: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/au

In [ ]:
import os
import json
import cv2
import subprocess


VIDEO_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4"

MAPPING_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/track_speaker_mapping.json"
OUTPUT_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips"

os.makedirs(OUTPUT_DIR, exist_ok=True)


with open(MAPPING_FILE, "r", encoding="utf-8") as f:
    mapping = json.load(f)

In [ ]:
cap = cv2.VideoCapture(VIDEO_FILE)

VIDEO_FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

cap.release()


print("FPS:", VIDEO_FPS)
print("Total frames:", TOTAL_FRAMES)

FPS: 25.011324024656833
Total frames: 1270


In [ ]:
for item in mapping:

    track_id = item["track_id"]
    segment_id = item["segment_id"]

    start_time = item["audio_start"]
    end_time = item["audio_end"]


    output_file = os.path.join(
        OUTPUT_DIR,
        f"track_{track_id}_segment_{segment_id}.mp4"
    )


    cmd = [
        "ffmpeg",
        "-y",

        "-i",
        VIDEO_FILE,

        "-ss",
        str(start_time),

        "-to",
        str(end_time),

        "-c:v",
        "libx264",

        "-c:a",
        "aac",

        "-preset",
        "fast",

        output_file,

        "-loglevel",
        "error"
    ]


    subprocess.run(cmd, check=True)


    print(
        f"Created {output_file} "
        f"({start_time:.3f}s -> {end_time:.3f}s)"
    )


print("All video clips extracted.")

Created /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_1.mp4 (0.900s -> 3.200s)
Created /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_2.mp4 (3.200s -> 4.300s)
Created /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_3.mp4 (4.800s -> 6.300s)
Created /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_4.mp4 (6.600s -> 9.900s)
Created /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_5.mp4 (9.900s -> 13.600s)
Created /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_6.mp4 (13.900s -> 16.800s)
Created /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_7.mp4 (17.000s -> 25.400s)
Created /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_7.mp4 (17.000s -> 25.400s)
Created /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips

# Face Clipping

In [ ]:
import os
import json
import cv2
import subprocess
import numpy as np

VIDEO_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4"

MAPPING_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/track_speaker_mapping.json"
OUTPUT_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips"


ACTIVE_SPEAKER_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/active_speakers.json"
MAPPING_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/track_speaker_mapping.json"

AUDIO_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/final_dubbed.wav"


FACE_CLIPS_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips"
AUDIO_CLIPS_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips"
OUTPUT_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/datawav2lip_results"


os.makedirs(FACE_CLIPS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


with open(ACTIVE_SPEAKER_FILE, "r") as f:
    active_data = json.load(f)


with open(MAPPING_FILE, "r") as f:
    mapping = json.load(f)

In [ ]:
TRACKING_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/tracks.json"


with open(TRACKING_FILE, "r") as f:
    tracking_data = json.load(f)

In [ ]:
tracks_bbox = {}

for item in tracking_data:

    tid = item["track_id"]

    if tid not in tracks_bbox:
        tracks_bbox[tid] = []

    tracks_bbox[tid].append(
        {
            "frame_idx": item["frame_idx"],
            "bbox": item["bbox"]
        }
    )

In [ ]:
import cv2
import os


def create_face_clip(
        video_path,
        track_frames,
        start_time,
        end_time,
        output_path
):

    cap = cv2.VideoCapture(video_path)


    fps = cap.get(
        cv2.CAP_PROP_FPS
    )


    start_frame = int(
        start_time * fps
    )

    end_frame = int(
        end_time * fps
    )


    # mapping frame -> bbox
    frame_map = {
        x["frame_idx"]:x["bbox"]
        for x in track_frames
    }


    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (224,224)
    )


    current_frame = 0


    while True:

        ret, frame = cap.read()

        if not ret:
            break


        if current_frame > end_frame:
            break



        if current_frame >= start_frame:


            if current_frame in frame_map:


                x1,y1,x2,y2 = map(
                    int,
                    frame_map[current_frame]
                )


                h,w,_ = frame.shape


                # safety
                x1=max(0,x1)
                y1=max(0,y1)

                x2=min(w,x2)
                y2=min(h,y2)


                face = frame[
                    y1:y2,
                    x1:x2
                ]


                if face.size > 0:

                    face = cv2.resize(
                        face,
                        (224,224)
                    )


                    writer.write(face)



        current_frame +=1


    cap.release()
    writer.release()

In [ ]:
for item in mapping:


    track_id = item["track_id"]

    segment_id = item["segment_id"]


    start_time = item["video_start"]

    end_time = item["video_end"]



    output_path = os.path.join(
        FACE_CLIPS_DIR,
        f"track_{track_id}_segment_{segment_id}.mp4"
    )


    print(
        "Processing:",
        output_path
    )


    create_face_clip(
        VIDEO_FILE,
        tracks_bbox[track_id],
        start_time,
        end_time,
        output_path
    )


print("Finished extracting face clips")

Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_1.mp4
Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_2.mp4
Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_3.mp4
Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_4.mp4
Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_5.mp4
Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_6.mp4
Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_7.mp4
Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_7.mp4
Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_8.mp4
Processing: /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_2_segment_9.mp4
Processing

# Combing face clip and audio clip

In [ ]:
import os
import json
import subprocess


MAPPING_FILE = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/track_speaker_mapping.json"

DUBBED_AUDIO = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/final_dubbed.wav"

FACE_CLIPS_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips"

AUDIO_CLIPS_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips"


os.makedirs(
    AUDIO_CLIPS_DIR,
    exist_ok=True
)

In [ ]:
with open(MAPPING_FILE, "r") as f:
    mapping = json.load(f)


print("Number of segments:", len(mapping))

print(mapping[0])

Number of segments: 12
{'track_id': 2, 'speaker_id': 'SPEAKER_01', 'segment_id': 1, 'video_start': 1.04, 'video_end': 2.159, 'audio_start': 0.9, 'audio_end': 3.2, 'overlap': 1.119}


In [ ]:
import os

AUDIO_CLIPS_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips"

# 1. Build the audio_files dictionary
audio_files = {}
for item in mapping:
    t_id = item["track_id"]
    s_id = item["segment_id"]
    audio_files[(t_id, s_id)] = os.path.join(
        AUDIO_CLIPS_DIR,
        f"track_{t_id}_segment_{s_id}.wav"
    )

# 2. Your existing loop
wav2lip_jobs = []

for item in mapping:
    track_id = item["track_id"]
    segment_id = item["segment_id"]

    job = {
        "track_id": track_id,
        "segment_id": segment_id,

        "face_video": os.path.join(
            FACE_CLIPS_DIR,
            f"track_{track_id}_segment_{segment_id}.mp4"
        ),

        "audio": audio_files[(track_id, segment_id)]
    }

    wav2lip_jobs.append(job)

In [ ]:
with open(
    "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/wav2lip_jobs.json",
    "w"
) as f:

    json.dump(
        wav2lip_jobs,
        f,
        indent=4
    )


print("wav2lip_jobs.json created")

wav2lip_jobs.json created


# Lip sync

In [ ]:
!cd /content/ && git clone https://github.com/Rudrabha/Wav2Lip.git

Cloning into 'Wav2Lip'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 409 (delta 2), reused 0 (delta 0), pack-reused 405 (from 2)
Receiving objects: 100% (409/409), 549.05 KiB | 1.79 MiB/s, done.
Resolving deltas: 100% (227/227), done.


In [ ]:
!mkdir -p checkpoints
!wget "https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth" -O "checkpoints/wav2lip_gan.pth"

--2026-07-12 21:55:14--  https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth
Resolving huggingface.co (huggingface.co)... 18.164.174.17, 18.164.174.23, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.17|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/63e453e727da6c56512c7024/40dfad7cf864d28deb16c06d9f37ebdff7fe14b063f0b162c2e9f3de4036d822?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27wav2lip_gan.pth%3B+filename%3D%22wav2lip_gan.pth%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1783896914&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjNlNDUzZTcyN2RhNmM1NjUxMmM3MDI0LzQwZGZhZDdjZjg2NGQyOGRlYjE2YzA2ZDlmMzdlYmRmZjdmZTE0YjA2M2YwYjE2MmMyZTlmM2RlNDAzNmQ4MjJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxlbmFtZSUyQSUzRFVURi04JTI3JTI3d2F2MmxpcF9nYW4ucHRoJTNCK2ZpbGVuYW1lJTNEJTIyd2F2MmxpcF9n

In [ ]:
!wget "https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth" -O "checkpoints/wav2lip_gan.pth"

--2026-07-12 21:55:21--  https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.17, 18.164.174.118, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/63e453e727da6c56512c7024/40dfad7cf864d28deb16c06d9f37ebdff7fe14b063f0b162c2e9f3de4036d822?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27wav2lip_gan.pth%3B+filename%3D%22wav2lip_gan.pth%22%3B&X-Xet-Cas-Uid=public&Expires=1783896921&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjNlNDUzZTcyN2RhNmM1NjUxMmM3MDI0LzQwZGZhZDdjZjg2NGQyOGRlYjE2YzA2ZDlmMzdlYmRmZjdmZTE0YjA2M2YwYjE2MmMyZTlmM2RlNDAzNmQ4MjJcXD91c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGl

In [ ]:
!pip install opencv-python-headless==4.8.0.74

Reason for being yanked: deprecated, use 4.8.0.76
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 13.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, but you have opencv-python-headless 4.8.0.74 which is incompatible.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, but you have opencv-python-headless 4.8.0.74 which is incompatible.


In [ ]:
!pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have nump

In [ ]:
!pip install librosa==0.9.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.3/214.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.6 MB/s eta 0:00:00
  Attempting uninstall: librosa
    Found existing installation: librosa 0.11.0
    Uninstalling librosa-0.11.0:
      Successfully uninstalled librosa-0.11.0


In [ ]:
# 1. Create the missing folder on your Google Drive
!mkdir -p /content/drive/MyDrive/Dubly_ME_Storage/data/wav2lip_results/

# 2. Run the Wav2Lip inference
!cd /content/Wav2Lip && python inference.py \
--checkpoint_path checkpoints/wav2lip_gan.pth \
--face /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_10.mp4 \
--audio /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_10.wav \
--outfile /content/drive/MyDrive/Dubly_ME_Storage/data/wav2lip_results/test_synced10.mp4

Using cuda for inference.
Reading video frames...
Number of frames available for inference: 173
/content/Wav2Lip/audio.py:100: FutureWarning: Pass sr=16000, n_fft=800 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,
(80, 553)
Length of mel chunks: 170
  0% 0/2 [00:00<?, ?it/s]
  0% 0/11 [00:00<?, ?it/s]
  9% 1/11 [00:54<09:01, 54.16s/it]
 18% 2/11 [00:57<03:37, 24.16s/it]
 27% 3/11 [01:00<01:55, 14.49s/it]
 36% 4/11 [01:03<01:10, 10.03s/it]
 45% 5/11 [01:06<00:45,  7.55s/it]
 55% 6/11 [01:10<00:30,  6.14s/it]
 64% 7/11 [01:12<00:20,  5.09s/it]
 73% 8/11 [01:16<00:13,  4.54s/it]
 82% 9/11 [01:19<00:08,  4.07s/it]
 91% 10/11 [01:22<00:03,  3.70s/it]
100% 11/11 [01:48<00:00,  9.90s/it]
Load checkpoint from: checkpoints/wav2lip_gan.pth
Model loaded
100% 2/2 [02:07<00:00, 63.86s/it]
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  bu

In [ ]:
import os
import subprocess


WAV2LIP_DIR = "/content/Wav2Lip"

CHECKPOINT = "checkpoints/wav2lip_gan.pth"


FACE_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips"

AUDIO_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips"


OUTPUT_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/wav2lip_results"


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


face_files = sorted(
    [
        f for f in os.listdir(FACE_DIR)
        if f.endswith(".mp4")
    ]
)


for face_file in face_files:

    name = face_file.replace(".mp4","")

    face_path = os.path.join(
        FACE_DIR,
        face_file
    )


    audio_path = os.path.join(
        AUDIO_DIR,
        name + ".wav"
    )


    if not os.path.exists(audio_path):
        print(
            "Missing audio:",
            audio_path
        )
        continue


    output_path = os.path.join(
        OUTPUT_DIR,
        name + "_synced.mp4"
    )


    cmd = f"""
    cd {WAV2LIP_DIR} &&
    python inference.py \
    --checkpoint_path {CHECKPOINT} \
    --face "{face_path}" \
    --audio "{audio_path}" \
    --outfile "{output_path}"
    """


    print("Processing:", name)


    subprocess.run(
        cmd,
        shell=True,
        check=True
    )


print("All clips finished")

Processing: track_2_segment_1
Processing: track_2_segment_10
Processing: track_2_segment_2
Processing: track_2_segment_3
Processing: track_2_segment_4
Processing: track_2_segment_5
Processing: track_2_segment_6
Processing: track_2_segment_7


CalledProcessError: Command '
    cd /content/Wav2Lip &&
    python inference.py     --checkpoint_path checkpoints/wav2lip_gan.pth     --face "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/video_clips/track_2_segment_7.mp4"     --audio "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_2_segment_7.wav"     --outfile "/content/drive/MyDrive/Dubly_ME_Storage/data/wav2lip_results/track_2_segment_7_synced.mp4"
    ' returned non-zero exit status 1.

In [ ]:
import cv2

video="/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_1_segment_6.mp4"

cap=cv2.VideoCapture(video)

length=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fps=cap.get(cv2.CAP_PROP_FPS)

print("Frames:", length)
print("FPS:", fps)

cap.release()

Frames: 88
FPS: 30.0


In [ ]:
import soundfile as sf

audio="/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_1_segment_6.wav"

data, rate = sf.read(audio)

print(data.shape)
print(rate)

(91200,)
16000


In [ ]:
!cd /content/Wav2Lip && python inference.py \
--checkpoint_path checkpoints/wav2lip_gan.pth \
--face "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/face_clips/track_1_segment_6.mp4" \
--audio "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/audio_clips/track_1_segment_6.wav" \
--nosmooth \
--pads 0 10 0 0 \
--outfile "/content/test.mp4"

Using cuda for inference.
Reading video frames...
Number of frames available for inference: 88
/content/Wav2Lip/audio.py:100: FutureWarning: Pass sr=16000, n_fft=800 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,
(80, 457)
Length of mel chunks: 167
  0% 0/2 [00:00<?, ?it/s]
  0% 0/6 [00:00<?, ?it/s]
 17% 1/6 [00:03<00:16,  3.39s/it]
 33% 2/6 [00:03<00:06,  1.55s/it]
 50% 3/6 [00:03<00:02,  1.02it/s]
 67% 4/6 [00:04<00:01,  1.40it/s]
 83% 5/6 [00:04<00:00,  1.82it/s]
100% 6/6 [00:05<00:00,  1.02it/s]
  0% 0/2 [00:06<?, ?it/s]
Traceback (most recent call last):
  File "/content/Wav2Lip/inference.py", line 280, in <module>
    main()
  File "/content/Wav2Lip/inference.py", line 249, in main
    for i, (img_batch, mel_batch, frames, coords) in enumerate(tqdm(gen, 
                                                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.

# Video Reconstruction

In [ ]:
import json
import cv2
import os

MAPPING_FILE ="/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/track_speaker_mapping.json"
WAV2LIP_RESULTS_DIR = "/content/drive/MyDrive/Dubly_ME_Storage/data/wav2lip_results"
ORIGINAL_VIDEO_PATH = "/content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/source_media.mp4"
OUTPUT_VIDEO_PATH = "/content/drive/MyDrive/Dubly_ME_Storage/data/merged_video_no_audio.mp4"

with open(MAPPING_FILE, "r") as f:
    mapping_data = json.load(f)

if isinstance(mapping_data, dict):
    mapping_data = [mapping_data]

cap_orig = cv2.VideoCapture(ORIGINAL_VIDEO_PATH)
fps = cap_orig.get(cv2.CAP_PROP_FPS)
width = int(cap_orig.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap_orig.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap_orig.get(cv2.CAP_PROP_FRAME_COUNT))

frame_sources = [None] * (total_frames + 1)
for item in mapping_data:
    track_id = item["track_id"]
    segment_id = item["segment_id"]

    start_frame = int(item["video_start"] * fps)
    end_frame = int(item["video_end"] * fps)

    video_name = f"synced_{track_id}_{segment_id}.mp4"

    for f_idx in range(start_frame, end_frame):
        if f_idx < len(frame_sources):
            frame_sources[f_idx] = video_name

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

current_synced_cap = None
current_synced_name = None

print("Starting video merge process...")

for i in range(total_frames):
    ret, orig_frame = cap_orig.read()
    if not ret:
        break

    source_vid = frame_sources[i]

    if source_vid is None:
        out.write(orig_frame)

        if current_synced_cap is not None:
            current_synced_cap.release()
            current_synced_cap = None
            current_synced_name = None
    else:
        if current_synced_name != source_vid:
            if current_synced_cap is not None:
                current_synced_cap.release()

            synced_path = os.path.join(WAV2LIP_RESULTS_DIR, source_vid)
            current_synced_cap = cv2.VideoCapture(synced_path)
            current_synced_name = source_vid

        ret_sync, sync_frame = current_synced_cap.read()
        if ret_sync:
            out.write(sync_frame)
        else:
            out.write(orig_frame)

cap_orig.release()
if current_synced_cap is not None:
    current_synced_cap.release()
out.release()

print(f"Merge complete! Video saved to {OUTPUT_VIDEO_PATH}")

Starting video merge process...
Merge complete! Video saved to /content/drive/MyDrive/Dubly_ME_Storage/data/merged_video_no_audio.mp4


In [ ]:
!ffmpeg -i /content/drive/MyDrive/Dubly_ME_Storage/data/merged_video_no_audio.mp4 -i /content/drive/MyDrive/Dubly_ME_Storage/data/audio_in/final_dubbed.wav -c:v copy -c:a aac -strict experimental /content/drive/MyDrive/Dubly_ME_Storage/data/final_output_video.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab